# 12 · Machine Learning básico: ¿el dólar y los combustibles explican la inflación?

Un ejemplo corto y completo del flujo de *machine learning* supervisado con **scikit-learn**, usando datos reales de la plataforma:

1. **Preparar los datos**: armar una tabla mensual con la variable a predecir y las variables explicativas.
2. **Dividir** en entrenamiento y prueba, respetando el orden del tiempo.
3. **Entrenar** una regresión lineal.
4. **Evaluar** con métricas de error y compararla con un modelo de referencia muy simple.
5. **Interpretar** y decir con honestidad qué no puede hacer el modelo.

**Pregunta de negocio:** ¿cuánto de la inflación interanual de RD se explica con la variación interanual del dólar y de los combustibles?

> Este notebook no es un modelo de pronóstico para producción. Es un ejercicio para mostrar que conozco el flujo y sé evaluarlo con cuidado.

**Cómo correrlo:** Serverless → **Run all**.

## 1 · Preparar los datos

| Variable | Rol | Fuente (Gold) | Cálculo |
|---|---|---|---|
| `inflacion_yoy` | **Objetivo (y)** | `inflation_metrics`, IPC general | Inflación interanual publicada por mes, % |
| `dolar_yoy` | Explicativa (X) | `exchange_rate_monthly_metrics`, venta | Variación % de la tasa promedio del mes contra el mismo mes del año anterior |
| `gasolina_yoy` | Explicativa (X) | `fuel_price_metrics`, gasolina regular | Precio promedio del mes, variación % interanual |
| `gasoil_yoy` | Explicativa (X) | `fuel_price_metrics`, gasoil regular | Igual que la gasolina |

Se filtra y se promedia en Spark; a pandas solo llega una fila por mes.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

GOLD = "rd_economic_intelligence.gold"

inflacion = (spark.table(f"{GOLD}.inflation_metrics")
             .filter("series_type = 'headline'")
             .select(F.col("reference_month").alias("mes"), F.col("yoy_inflation_pct").cast("double").alias("inflacion_yoy")))

dolar = (spark.table(f"{GOLD}.exchange_rate_monthly_metrics")
         .filter("rate_type = 'sell'")
         .select(F.col("month_start").alias("mes"), F.col("month_avg_rate").cast("double").alias("dolar")))

combustibles = (spark.table(f"{GOLD}.fuel_price_metrics")
                .filter("fuel_code IN ('gasolina_regular', 'gasoil_regular')")
                .withColumn("mes", F.trunc("period_start_date", "month"))
                .groupBy("mes").pivot("fuel_code", ["gasolina_regular", "gasoil_regular"])
                .agg(F.avg(F.col("price_dop").cast("double")))
                .withColumnRenamed("gasolina_regular", "gasolina").withColumnRenamed("gasoil_regular", "gasoil"))

mensual = (dolar.join(combustibles, "mes", "inner").join(inflacion, "mes", "inner")
           .orderBy("mes").toPandas())
mensual["mes"] = pd.to_datetime(mensual["mes"])
print("Meses con los tres datos:", len(mensual), "| desde", mensual["mes"].min().date(), "hasta", mensual["mes"].max().date())

In [0]:
# pct_change(12) compara con 12 filas atrás: solo es correcto si no falta ningún mes.
meses_esperados = pd.date_range(mensual["mes"].min(), mensual["mes"].max(), freq="MS")
assert len(meses_esperados) == len(mensual), "Faltan meses: pct_change(12) compararía contra el mes equivocado"

df = mensual.set_index("mes")
for col in ["dolar", "gasolina", "gasoil"]:
    df[f"{col}_yoy"] = df[col].pct_change(12) * 100

df = df.dropna(subset=["dolar_yoy", "gasolina_yoy", "gasoil_yoy", "inflacion_yoy"])
print("Filas para el modelo:", len(df), "| desde", df.index.min().date(), "hasta", df.index.max().date())
display(df.reset_index().tail(6).round(2))

**Detalle que vale la pena contar:** antes de calcular `pct_change(12)` se comprueba que no falte ningún mes. Si faltara uno, "12 filas atrás" ya no sería "el mismo mes del año anterior" y el cálculo quedaría mal sin dar ningún error.

## 2 · Explorar: correlación

La correlación (de −1 a 1) mide qué tan juntas se mueven dos variables. Es un primer filtro, no una prueba de causa.

In [0]:
corr = df[["inflacion_yoy", "dolar_yoy", "gasolina_yoy", "gasoil_yoy"]].corr().round(2)
display(corr.reset_index().rename(columns={"index": "variable"}))

**Lectura (190 meses, 2011–2026):**

- Gasolina y gasoil se mueven casi igual con la inflación: correlación de **0.78** y **0.76**.
- Gasolina y gasoil entre sí: **0.97**. Son casi la misma información; esto importa al leer los coeficientes.
- El dólar tiene una correlación **negativa (−0.34)** con la inflación en esta muestra. Va en contra de la idea de que "si sube el dólar, sube la inflación". Los datos muestran por qué: en 2020 el dólar subió 10.4 % en promedio interanual y la inflación fue 3.8 %; en 2022 el dólar bajó 3.7 % y la inflación llegó a 8.8 %, empujada por combustibles y precios internacionales. No lo interpreto como causa; lo anoto como algo que revisar.

## 3 · Dividir en entrenamiento y prueba (respetando el tiempo)

En datos que dependen del tiempo **no se mezclan las filas al azar** (`train_test_split(shuffle=True)` sería un error). Si mezclo, el modelo entrena con meses de 2024 y se evalúa con meses de 2023: estaría "viendo el futuro" (*data leakage*) y las métricas saldrían mejores de lo real.

Corte usado: **entrenamiento hasta diciembre de 2022, prueba desde enero de 2023.**

In [0]:
X_COLS = ["dolar_yoy", "gasolina_yoy", "gasoil_yoy"]
CORTE = "2022-12-01"

train, test = df.loc[:CORTE], df.loc[CORTE:].iloc[1:]
X_train, y_train = train[X_COLS], train["inflacion_yoy"]
X_test, y_test = test[X_COLS], test["inflacion_yoy"]
print(f"Entrenamiento: {len(train)} meses ({train.index.min():%Y-%m} a {train.index.max():%Y-%m})")
print(f"Prueba:        {len(test)} meses ({test.index.min():%Y-%m} a {test.index.max():%Y-%m})")

## 4 · Entrenar y evaluar

Se comparan dos modelos:

- **Referencia (baseline):** predice siempre la inflación promedio del periodo de entrenamiento. Si el modelo no le gana a esto, no aporta nada.
- **Regresión lineal:** `inflación = b0 + b1·dólar + b2·gasolina + b3·gasoil`.

**Métricas** (en puntos porcentuales de inflación):

| Métrica | Qué dice | Mejor si… |
|---|---|---|
| **MAE** | Error promedio, en las mismas unidades | más bajo |
| **RMSE** | Como el MAE, pero castiga más los errores grandes | más bajo |
| **R²** | Qué parte de la variación explica el modelo (1 = todo, 0 = igual que el promedio, negativo = peor que el promedio) | más alto |

In [0]:
def evaluar(nombre, y_real, y_pred):
    return {"modelo": nombre,
            "MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
            "R2": r2_score(y_real, y_pred)}

# Referencia: el promedio de entrenamiento
pred_base = np.full(len(y_test), y_train.mean())

# Regresión lineal
modelo = LinearRegression()
modelo.fit(X_train, y_train)
pred_lr = modelo.predict(X_test)

resultados = pd.DataFrame([evaluar("Referencia: promedio de entrenamiento", y_test, pred_base),
                           evaluar("Regresión lineal (dólar + combustibles)", y_test, pred_lr),
                           {**evaluar("Regresión lineal, sobre entrenamiento", y_train, modelo.predict(X_train)), "modelo": "Regresión lineal, sobre entrenamiento"}])
display(resultados.round(3))

**Lectura:**

- En prueba (2023-01 a 2026-10, 46 meses), la regresión lineal tiene un **MAE de 0.59** puntos contra **0.81** del promedio simple. En promedio se equivoca por poco más de medio punto de inflación, y le gana a la referencia.
- El **R² en prueba es bajo (0.17)**, aunque en entrenamiento fue **0.64**. Es normal que baje fuera de la muestra; además, en 2023–2026 la inflación varió poco (entre 3 % y 7 %), y con poca variación el R² tiende a ser pequeño aunque el error sea bajo. Por eso se reportan varias métricas y no solo una.
- Que el error de entrenamiento (MAE 1.31) sea **mayor** que el de prueba no es una señal de que el modelo sea excelente: el entrenamiento incluye el salto de inflación de 2021–2022, mucho más difícil de explicar.

## 5 · Interpretar los coeficientes

In [0]:
coef = pd.DataFrame({"variable": ["intercepto (b0)"] + X_COLS,
                     "coeficiente": [modelo.intercept_] + list(modelo.coef_)}).round(4)
display(coef)

**Lectura de los coeficientes** (cada uno, manteniendo las otras variables fijas):

- **Gasolina (+0.16):** por cada 10 puntos de alza interanual del precio de la gasolina, el modelo asocia unos 1.6 puntos más de inflación.
- **Gasoil (−0.03):** sale negativo, lo que no tiene sentido económico. Es el efecto de la multicolinealidad: como gasolina y gasoil se mueven igual (0.97), el modelo le asigna casi todo el efecto a la gasolina y deja al gasoil con un número pequeño de signo cualquiera. Por eso no se lee cada coeficiente como un efecto exacto.
- **Dólar (−0.08):** negativo, igual que la correlación. Es una asociación en esta muestra, no el efecto de traspaso del tipo de cambio que describe la teoría.
- **Intercepto (3.78):** la inflación estimada cuando las tres variaciones son cero; cercana a la meta del BCRD de 4 % ± 1 %.

In [0]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df.index, df["inflacion_yoy"], label="Inflación real", color="#1f4e79", linewidth=2)
ax.plot(train.index, modelo.predict(X_train), label="Modelo (entrenamiento)", color="#7f7f7f", linestyle="--")
ax.plot(test.index, pred_lr, label="Modelo (prueba)", color="#c55a11", linestyle="--", linewidth=2)
ax.axvline(pd.Timestamp("2023-01-01"), color="black", linewidth=0.8)
ax.text(pd.Timestamp("2023-02-01"), ax.get_ylim()[1] * 0.92, "inicio de la prueba", fontsize=9)
ax.set_ylabel("Inflación interanual, %")
ax.set_title("Inflación real vs. regresión lineal con dólar y combustibles")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.show()

## 6 · Experimento: agregar la inflación del mes anterior

Un ajuste común en series de tiempo: incluir el valor del mes anterior (*lag*). Es legítimo si el objetivo es estimar el mes siguiente, porque ese dato ya se conoce cuando se hace la estimación.

In [0]:
df["inflacion_mes_anterior"] = df["inflacion_yoy"].shift(1)
d2 = df.dropna(subset=["inflacion_mes_anterior"])
X2 = X_COLS + ["inflacion_mes_anterior"]
tr2, te2 = d2.loc[:CORTE], d2.loc[CORTE:].iloc[1:]

modelo2 = LinearRegression().fit(tr2[X2], tr2["inflacion_yoy"])
pred2 = modelo2.predict(te2[X2])
pred_ingenuo = te2["inflacion_mes_anterior"]          # "el mes que viene será igual a este"

comparacion = pd.DataFrame([evaluar("Ingenuo: igual al mes anterior", te2["inflacion_yoy"], pred_ingenuo),
                            evaluar("Regresión con mes anterior", te2["inflacion_yoy"], pred2)])
display(comparacion.round(3))
display(pd.DataFrame({"variable": ["intercepto (b0)"] + X2, "coeficiente": [modelo2.intercept_] + list(modelo2.coef_)}).round(4))

**Lectura:**

- El modelo **ingenuo** ("el mes que viene será igual a este") ya tiene un MAE de **0.25** y un R² de **0.88**. Es mucho mejor que la regresión con solo dólar y combustibles.
- Con el mes anterior incluido, la regresión baja a un MAE de **0.20** (R² **0.93**). Mejora al ingenuo, pero poco.
- El coeficiente del mes anterior es **0.92**: casi toda la estimación viene de la propia inercia de la inflación; dólar y combustibles agregan un ajuste pequeño.

**Conclusión:** la inflación es muy persistente. Cualquier modelo de inflación mensual debe compararse contra el ingenuo, porque es una referencia muy difícil de superar. Saber esto y decirlo vale más en una entrevista que mostrar un R² alto.

## 7 · Lo que este modelo no puede hacer (dilo tú antes que te lo pregunten)

- **Pocos datos:** son meses, no millones de filas. Unas decenas de observaciones en la prueba no alcanzan para conclusiones firmes.
- **Correlación no es causa:** el dólar y los combustibles se mueven junto con otras cosas (precios internacionales, política monetaria, choques como 2021–2022) que el modelo no ve.
- **Variables parecidas entre sí:** gasolina y gasoil suben y bajan casi juntas (multicolinealidad). El modelo puede repartir el efecto entre las dos de forma inestable, así que un coeficiente individual no se debe leer como "el efecto exacto" de ese combustible.
- **El efecto llega con retraso:** un alza del combustible tarda meses en pasar a los precios. El modelo usa el mismo mes.
- **Las relaciones cambian con el tiempo:** lo que valía en 2012–2022 no tiene por qué valer igual después.

**Siguientes pasos razonables:** validación cruzada para series de tiempo (`TimeSeriesSplit`), probar rezagos de 3 a 6 meses en dólar y combustibles, y regularización (`Ridge`) por la multicolinealidad. Y siempre comparar contra el modelo ingenuo.

## 8 · Cómo contarlo en la entrevista (30 segundos)

> "Tengo un ejemplo pequeño con scikit-learn sobre mis datos: una regresión lineal de la inflación interanual con la variación del dólar y de los combustibles. Separé entrenamiento y prueba por fecha, sin mezclar, para no filtrar el futuro. Lo comparé contra un promedio simple y contra el modelo ingenuo de 'igual al mes anterior', y lo medí con MAE, RMSE y R². La conclusión honesta es que los combustibles y el dólar ayudan a explicar la tendencia, pero la mayor parte de la información está en la propia inercia de la inflación. No me presento como científico de datos, pero conozco el flujo y sé evaluar si un modelo realmente aporta."